In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici")
dbutils.widgets.text("esquema", "bronze")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

In [0]:
weather_schema = StructType([
    StructField("date", DateType(), True),
    StructField("temperature_max", DoubleType(), True),
    StructField("temperature_avg", DoubleType(), True),
    StructField("precipitation", DoubleType(), True)
])

In [0]:
dates_df = spark.read.table(f"{catalogo}.{esquema}.ecobici") \
    .select(min("fecha").alias("min_date"), max("fecha").alias("max_date"))


In [0]:
dates = dates_df.collect()[0]
start_date = str(dates["min_date"])
end_date = str(dates["max_date"])


In [0]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 19.4326,
    "longitude": -99.1332,
    "start_date": start_date,
    "end_date": end_date,
    "daily": "temperature_2m_max,temperature_2m_mean,precipitation_sum",
    "timezone": "America/Mexico_City"
}
response = requests.get(url, params=params, timeout=60)
response.raise_for_status()
data = response.json()


In [0]:
weather_data = list(zip(
    data["daily"]["time"],
    data["daily"]["temperature_2m_max"],
    data["daily"]["temperature_2m_mean"],
    data["daily"]["precipitation_sum"]
))


In [0]:
df = spark.createDataFrame(weather_data, ["date", "temperature_max", "temperature_avg", "precipitation"])


In [0]:
df_final = df \
    .withColumn("date", to_date(col("date"))) \
    .withColumn("ingestion_date", current_timestamp())


In [0]:
df_final.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema}.weather")

In [0]:
%sql
SELECT COUNT(*) FROM catalog_ecobici.bronze.weather;

In [0]:
%sql
SELECT *
FROM catalog_ecobici.bronze.weather
LIMIT 10;

In [0]:
%sql
SELECT 
    MIN(date) AS fecha_inicio,
    MAX(date) AS fecha_fin
FROM catalog_ecobici.bronze.weather;

In [0]:
%sql
SELECT COUNT(*) AS registros_match
FROM catalog_ecobici.bronze.ecobici e
JOIN catalog_ecobici.bronze.weather w
ON e.fecha = w.date;

In [0]:
%sql
SELECT *
FROM catalog_ecobici.bronze.weather
WHERE temperature_max IS NULL
   OR temperature_avg IS NULL
   OR precipitation IS NULL;

In [0]:
%sql
SELECT 
    AVG(temperature_avg) AS temp_promedio,
    MAX(temperature_max) AS temp_maxima,
    SUM(precipitation) AS lluvia_total
FROM catalog_ecobici.bronze.weather;

In [0]:
%sql
SELECT date, COUNT(*) AS duplicados
FROM catalog_ecobici.bronze.weather
GROUP BY date
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT 
    DATEDIFF(MAX(date), MIN(date)) + 1 AS dias_esperados,
    COUNT(*) AS dias_reales
FROM catalog_ecobici.bronze.weather;